# 代入式を使い内包表記での繰り返し作業をなくす

リスト、辞書、集合などでの内包表記のよく使われるパターンはでは、同じ計算を複数個所で参照する必要があります。

例えば、ねじ締めなどの締結部材を扱う会社で注文管理のプログラムを考えます。顧客からの新規注文に対して、注文に応じられるか答える必要があります。注文に応じられるだけの在庫があるか、出荷する最低個数（8個）以上かを確認する必要があります。

In [2]:
stock = {
  'nails': 125,
  'screws': 35,
  'wingnuts': 8,
  'washers': 24,
}

order = ['screws', 'wingnuts', 'clips']

def get_batches(count, size):
  print(f'{count} // {size} = {count // size}')
  return count // size # 切り捨て除算

result = {}
for name in order:
  count = stock.get(name, 0)
  batches = get_batches(count, 8)
  if batches:
    result[name] = batches

print(result)

35 // 8 = 4
8 // 8 = 1
0 // 8 = 0
{'screws': 4, 'wingnuts': 1}


次のコードは、辞書内包表記を使ってループのロジックをより簡潔に実装しています。

In [ ]:
found = {name: get_batches(stock.get(name, 0), 8) for name in order 
         if get_batches(stock.get(name, 0), 8)}
print(found)

35 // 8 = 4
35 // 8 = 4
8 // 8 = 1
8 // 8 = 1
0 // 8 = 0
{'screws': 4, 'wingnuts': 1}


Python の辞書内包表記は 左から右へ、ただし if 条件 は値の生成前に評価 されます。
1. for name in order の 最初の name を 1つ取り出す
1. stock.get(name, 0) を評価し、在庫数を取得
1. if get_batches(stock.get(name, 0), 8)
   - → get_batches() を呼び出し、8個で割った何セットになるかを計算
   - → 0 なら False とみなされるので除外
   - → 1以上なら True なので続行
1. name: get_batches(stock.get(name, 0), 8) を辞書に追加
   - → 辞書の値用にもう一度 get_batches() が呼ばれる
1. 次の name に進む

このコードは簡潔なのですが、式 get_batches(stock.get(name, 0), 8) が繰り返されるという問題があります。
これは、不必要な記述が目障りで読みやすさを損なっています。2つの指揮を同じにする必要があるので、バグの危険性が高くなります。

例えば、最初の get_batches だけ、第2引数を 8 ではなく 4 にすると、結果が異なってきます。

In [5]:
has_bug = {name: get_batches(stock.get(name, 0), 4) for name in order
           if get_batches(stock.get(name, 0), 8)}
print("Expected:", found)
print("Found:", has_bug)

35 // 8 = 4
35 // 4 = 8
8 // 8 = 1
8 // 4 = 2
0 // 8 = 0
Expected: {'screws': 4, 'wingnuts': 1}
Found: {'screws': 8, 'wingnuts': 2}


この問題の簡単な解決法は、Python 3.8 から導入されたウォルラス演算子を用いて、内包表記で代入式を使うことです。

In [ ]:

# 辞書内包表記を使うと簡潔になる
found = {name: batches
         for name in order
         if (batches:=get_batches(stock.get(name, 0), 8))}
print(found)

35 // 8 = 4
8 // 8 = 1
0 // 8 = 0
{'screws': 4, 'wingnuts': 1}


代入式（batches:=get_batches(...)）により、order キーそれぞれについて stock 辞書の検索と get_batches の呼び出しを一度行うだけで、結果を変数 batches に格納できました。

内包表記の値の式で代入式を定義するのは正しい構文ですが、定義している変数をその内包表記の別の場所で参照しようとすると、内包表記の評価される順序によっては、実行時例外が発生することがあります。

In [6]:
result = {name: (tenth := count // 10) for name, count in stock.items() if tenth > 0}

NameError: name 'tenth' is not defined

この例題では、代入式を条件中に移動し、定義した変数名を内包表記の値の部分で参照すればエラーをなくせます。

In [7]:
result = {name: tenth for name, count in stock.items() if (tenth := count // 10) > 0}

In [16]:
# 内包表記の中でウォルラス演算子を使うと、変数名が内包表記を含むスコープにリークする（項目21参照）
half = [(squared := last ** 2)
        for count in stock.values()
        if (last := count // 2) > 10]
print(f'Last item of {half} is {last} ** 2 = {squared}')

Last item of [3844, 289, 144] is 12 ** 2 = 144


In [17]:
# for文でも変数名のリークは発生
for count in stock.values():
  last = count // 2
  squared = last ** 2

print(f'{count} // 2 = {last}; {last} ** 2 = {squared}')

24 // 2 = 12; 12 ** 2 = 144


In [8]:
# しかし、代入分を使わない内包表記のループ変数ではリークが発生しない
half = [count // 2 for count in stock.values()]
print(half)
print(count) # ループ変数がリークしないので例外が発生

[62, 17, 4, 12]
0


In [19]:
# ジェネレータ式でも代入式が使える
found = ((name, batches) for name in order
         if (batches := get_batches(stock.get(name, 0), 8)))
print(next(found))
print(next(found))

35 // 8 = 4
('screws', 4)
8 // 8 = 1
('wingnuts', 1)


In [20]:
# ジェネレータ式でも代入式が使える
found = ({name: batches} for name in order
         if (batches := get_batches(stock.get(name, 0), 8)))
print(next(found))
print(next(found))

35 // 8 = 4
{'screws': 4}
8 // 8 = 1
{'wingnuts': 1}


In [ ]:
# タプル
a = (1, 2, 3) # a = 1, 2, 3 カッコなしでもカンマがあればタプル
print(a)
print(type(a))

# ジェネレータ ()内が内包表記
b = (x*2 for x in range(3))
print(b)
print(type(b))

(1, 2, 3)
<class 'tuple'>
<generator object <genexpr> at 0x7f9910d94110>
<class 'generator'>


## 覚えておくこと

- 代入式により、内包表記とジェネレータ式で内包表記の条件部分の値を再利用でき、読みやすさと性能が向上する
- 内包表記やジェネレータ式の条件部分以外で代入式を使うこともできるが、避けるべきだ